This notebook focuses on building a customer churn prediction model using XGBoost, combined with structured and reusable feature engineering techniques implemented through custom Scikit-learn transformers.

* Objective:

> To improve churn prediction accuracy by applying meaningful feature transformations and engineered features instead of relying only on raw data.

Hope this will be helpful!!!

## Importing data:

In [1]:
import pandas as pd

train_path = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
train_df = pd.read_csv(train_path)
train_df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [2]:
#looking for null values
train_df.isna().sum()

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

No null values present, we can proceed safely.

In [3]:
target = 'Churn'

#encoding the target 
train_df[target] = train_df[target].map({"No": 0, "Yes": 1}) 

#dropping redundant features ('id')
train_df = train_df.drop('id', axis=1)

#seperating features and label
X = train_df.drop(target, axis=1)
y = train_df[target]

## Feature Engineering:
Trying the best feature engineering methods to create new and more informative features. This section will consist:

* Binning 'tenure'
* Groupby categorical features and aggrigate statistics of numerical features

We'll create custom transformers using BaseEstimator and TransformerMixin, what ScikitLearn does is:

Call fit() on your transformer
Then immediately call transform() on that SAME training data, without an explicit call

The transformers i'm using here were created in the Heart Disease prediction competition, so we'll reuse the same code, though i'll still explin what they do...

### Binning transformer:

In [4]:
# from sklearn.base import BaseEstimator, TransformerMixin

# class Binning(BaseEstimator, TransformerMixin):
#     def __init__(self, col_to_bin, num_bins, new_col_name ,labels=None):
#         self.col_to_bin = col_to_bin
#         self.num_bins = num_bins
#         self.labels = labels
#         self.new_col_name = new_col_name

#     def fit(self, X, y=None):
#         X = X.copy()
#         _, self.bin_edges = pd.cut(X[self.col_to_bin], bins=self.num_bins, labels=False, retbins=True) #the last attribute ensures that the same bins are used to bin teh test data during transform
#         return self

#     def transform(self,X):
#         X = X.copy() 
#         X[self.new_col_name] = pd.cut(X[self.col_to_bin], bins=self.bin_edges, labels=False)
#         return X

The above binning transformer was discarded, it was causing some issues during preprocessing. Its still kept in the notebook to get the idea what should be avoided during creating custom transformers.

In [5]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class Binning(BaseEstimator, TransformerMixin):
    def __init__(self, col_to_bin, num_bins, new_col_name=None, labels=None):
        self.col_to_bin = col_to_bin  # can be name or index
        self.num_bins = num_bins
        self.labels = labels
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        X = self._ensure_dataframe(X)

        col = self._get_column(X)

        _, self.bin_edges_ = pd.cut(
            col,
            bins=self.num_bins,
            labels=False,
            retbins=True
        )

        return self

    def transform(self, X):
        X = self._ensure_dataframe(X)

        col = self._get_column(X)

        binned = pd.cut(
            col,
            bins=self.bin_edges_,
            labels=False
        )

        if self.new_col_name:
            X[self.new_col_name] = binned
        else:
            X[self.col_to_bin] = binned

        return X

    # -------- helper methods --------

    def _ensure_dataframe(self, X):
        if isinstance(X, np.ndarray):
            return pd.DataFrame(X)
        return X

    def _get_column(self, X):
        if isinstance(self.col_to_bin, int):
            return X.iloc[:, self.col_to_bin]
        else:
            return X[self.col_to_bin]

### GroupMean Encoder:
Applying group-mean encoding on Monthly and Total Charges columns, by grouping them using 'tenure bins' created using binning transformer

In [6]:
class GroupMeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, groupby_col, agg_col, new_col_name):
        self.groupby_col = groupby_col
        self.agg_col = agg_col
        self.new_col_name = new_col_name

    def fit(self,X,y=None):
        self.means = X.groupby(self.groupby_col,observed=True)[self.agg_col].mean()
        return self

    def transform(self,X):
        X = X.copy()
        X[self.new_col_name] = X[self.groupby_col].map(self.means)
        return X

More advance stuff like **target encoding**, **frequency encoding** can also be used, I wont use it right away since im trying to get the idea about which engineered features work and which are adding noise.

## Preprocessing Pipeline:
Our preprocessing pipeline will consist of following:
1. Binning of 'tenure'
2. GroupMeanEncoding on 'MonthlyCharges' and 'TotalCharges' groupingby 'tenure_bins'

Wont be performing any scaling or OneHot encoding sicne we are going to use tree based algorithm that is XGBoost for prediction so its not nescessary.

In [7]:
#The following approach isnt the best for this task, so try avoiding this insted .....
#use OneHotEncoder or a LabelEncoder

cat_cols = X.select_dtypes(include='object').columns
X[cat_cols] = X[cat_cols].astype('category')

In [8]:
cat_cols = X.select_dtypes(include="category").columns
num_cols = ["TotalCharges","MonthlyCharges"]

In [9]:
from sklearn.pipeline import Pipeline

preprocessor2 = Pipeline([
    ('binning', Binning(col_to_bin='tenure', num_bins=3, new_col_name='tenure_bins')),
    ('GroupMeanEncoding_MonthlyCharges', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='MonthlyCharges', new_col_name='X1')),
    ('GroupMeanEncoding_TotalCharges', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='TotalCharges', new_col_name='X2')),
])

## Optuna Tuning:
**Optuna** helps us find best hyperparameters for our model with respect to our current preprocessing pipeline and feature space, given the range/grid of hyperparameters. Its better than **GridSearchCV** since it deosnt try all possible values insted tries some specific values which it thinks will yield good results saves processing time and expense for all different trials that we would have had to go through using gridsearch.

### XGBoost:
Using **XGBoost** because it yielded better results in baseline experimentation, another reason is its GPU support which boosts training speed.

In [10]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):

    params = {
        # Core learning parameters
        "n_estimators": trial.suggest_int("n_estimators", 500, 3500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        
        # Tree complexity control
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        
        # Sampling 
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        
        # Regularization 
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),

        # Performance
        "random_state": 42,
        "eval_metric": "logloss",
        "tree_method": "hist",   
        "verbosity": 0,
        "device":'cuda',
        "enable_categorical":True
    }

    model = Pipeline([
        ('prep2', preprocessor2),
        ('XGB', XGBClassifier(**params))
    ])

    score = cross_val_score(model, X, y, cv=5, scoring="roc_auc").mean()

    return score

In [11]:
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=50)

In [12]:
# print(study.best_params)
# print(study.best_value)

* **best_parameters found** = {'n_estimators': 3488, 'learning_rate': 0.04215348921376158,                                         'max_depth':3, 'min_child_weight': 3, 'gamma': 0.373740919749451,                                    'subsample': 0.8908198617400997, 'colsample_bytree': 0.7125429511845258,                             'reg_alpha': 5.2763994068891605e-08, 'reg_lambda': 1.0023283765023319e-07}
* **best_score** = 0.9166891748146979

In [13]:
best_xgb_params = {'n_estimators': 3488, 
                   'learning_rate': 0.04215348921376158,
                   'max_depth':3, 
                   'min_child_weight': 3, 
                   'gamma': 0.373740919749451, 
                   'subsample': 0.8908198617400997, 
                   'colsample_bytree': 0.7125429511845258,
                   'reg_alpha': 5.2763994068891605e-08, 
                   'reg_lambda': 1.0023283765023319e-07}

## Retraining on fill traindata:

In [14]:
best_model = XGBClassifier(
    **best_xgb_params,
    random_state= 42,
    eval_metric= "logloss",
    tree_method= "hist",   
    verbosity= 0,
    device='cuda',
    enable_categorical=True
)

final_pipeline = Pipeline([
        ('prep2', preprocessor2),
        ('model', best_model)
    ])

final_pipeline.fit(X,y)

Pipeline(steps=[('prep2',
                 Pipeline(steps=[('binning',
                                  Binning(col_to_bin='tenure',
                                          new_col_name='tenure_bins',
                                          num_bins=3)),
                                 ('GroupMeanEncoding_MonthlyCharges',
                                  GroupMeanEncoder(agg_col='MonthlyCharges',
                                                   groupby_col='tenure_bins',
                                                   new_col_name='X1')),
                                 ('GroupMeanEncoding_TotalCharges',
                                  GroupMeanEncoder(agg_col='TotalCharges',
                                                   groupby_col='tenure_bins',
                                                   new_co...
                               gamma=0.373740919749451, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.04215348921376158, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=3,
                               max_leaves=None, min_child_weight=3, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=3488, n_jobs=None,
                               num_parallel_tree=None, ...))])

## Predicting on test data:

In [15]:
test_path = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
test_df = pd.read_csv(test_path)
test_df.head() #to check if the data has loaded safely

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [16]:
X_test = test_df.drop('id', axis=1)

In [17]:
cat_cols = X_test.select_dtypes(include='object').columns
X_test[cat_cols] = X_test[cat_cols].astype('category')

In [18]:
y_pred = final_pipeline.predict_proba(X_test)[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [19]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: y_pred
})

submission.to_csv('submission.csv', index=False)